# SIP Week 2 — Compare Your Calibration Data: *with line* vs *without line*

You'll analyze **your own** head-tracking calibration data. The website can record calibration **two ways**:

- **Without line** — the dot jumps to a small set of **fixed dots** (~20 positions).
- **With line** — the dot **slides smoothly** along a path, so every frame is a new position.

Record **both** (or use the included samples), then your job is to **compare them** and answer some **intuitive questions** about what's different and why.

### How to use this notebook
- Run each cell with **Shift+Enter**.
- Cells marked **✏️ Exercise** are for *you*. Replace the `# TODO` with your code and answer the questions.
- Start with the sample files, then point the paths at **your own** two files (Part 2).

> Tip: if a cell errors, read the **last line** of the error first.

## 📹 First: record your own two calibration files

Do the calibration **twice** on the website — once in each mode — so you end up with two CSVs to compare.

1. Open the calibration website and go to the **start / config** screen.
2. Find the **"Line Animation"** option (two radio buttons):
   - **"Without a growing line"** → the dot jumps between **fixed dots**.
   - **"With a growing line"** → the dot **slides** along a path.
3. Pick a mode, then do the calibration (**follow the dot with your head**). When it finishes, a file called **`calibration_video1_<numbers>.csv`** downloads automatically.
4. ⚠️ The filename does **not** say which mode it was — so **rename it right away**, e.g.:
   - without-line run → `my_calibration_dots.csv`
   - with-line run → `my_calibration_moving.csv`
5. Refresh the page and repeat for the **other** mode.
6. Put both files **in the same folder as this notebook**, then set the two paths in **Part 2**.

*No webcam / can't record right now? Just use the included `sample_calibration_*.csv` files and continue.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)
print("Ready! Run the next cells with Shift+Enter.")

## Part 1 — pandas refresher

A **DataFrame** is just a table: rows and named columns. Recall the everyday moves:
- `df.head()` — first rows · `df.shape` — (rows, columns) · `df.columns` — names
- `df["col"]` — one column · `df[df["col"] > x]` — filter rows
- `df.describe()` — quick stats · `df["col"].mean()`, `.std()`, `.min()`, `.max()`

In [ ]:
# A tiny demo DataFrame (made-up data)
demo = pd.DataFrame({
    "name":  ["Ava", "Ben", "Cy", "Dia", "Eli"],
    "age":   [21, 23, 22, 24, 20],
    "score": [88, 72, 95, 60, 79],
})

print("shape:", demo.shape)
display(demo.head())
print("mean score:", demo["score"].mean())
display(demo[demo["score"] >= 80])    # filter: high scorers

In [ ]:
# ✏️ Exercise 1 — warm up on the demo table above
# 1) Print only the "name" and "score" columns.
# 2) Find the average age.
# 3) Show the rows where age is less than 22.
# 4) What is the highest score? (use .max())

# TODO: your code here


## Part 2 — Load BOTH of your calibration files

Each **row** in a calibration file is one captured frame.

**Important detail:** the file's **first line** starts with `#` and holds metadata (screen size, etc.) as JSON. The real column names are on the **second line** — so we load it with `pd.read_csv(..., comment="#")`. The helper `load_calib()` below does this for you and also returns the metadata.

**Key columns:**
| Column | Meaning |
|---|---|
| `targetX`, `targetY` | where the on-screen dot was (pixels) |
| `targetXRel`, `targetYRel` | same, as a 0–1 fraction of the screen |
| `yaw` | head turn left/right (deg) |
| `pitch` | head tilt up/down (deg) |
| `roll` | head tilt sideways (deg) |
| `landmark*` | face landmark positions (ignore for now) |

👉 Point `WITH_LINE_PATH` and `WITHOUT_LINE_PATH` at **your own** two files (or keep the samples).

In [ ]:
def load_calib(path):
    """Load a calibration CSV. Returns (dataframe, metadata_dict)."""
    with open(path) as f:
        first_line = f.readline().strip()
    meta = json.loads(first_line.lstrip("#")) if first_line.startswith("#") else {}
    df = pd.read_csv(path, comment="#")
    return df, meta

# 👉 CHANGE THESE to YOUR two files:
WITH_LINE_PATH    = "sample_calibration_moving.csv"   # moving dot (with line)
WITHOUT_LINE_PATH = "sample_calibration_dots.csv"     # fixed dots (without line)

df_line, meta_line = load_calib(WITH_LINE_PATH)
df_dots, meta_dots = load_calib(WITHOUT_LINE_PATH)

print("WITH line  :", df_line.shape, "| screen", meta_line.get("calibrationWidth"), "x", meta_line.get("calibrationHeight"))
print("WITHOUT line:", df_dots.shape, "| screen", meta_dots.get("calibrationWidth"), "x", meta_dots.get("calibrationHeight"))
df_line.head()

In [ ]:
# ✏️ Exercise 2 — get to know BOTH files
# For df_line AND df_dots, find out:
#   1) How many rows (frames) does each have? (.shape)
#   2) Peek at the head-pose columns: df[["yaw","pitch","roll"]].head()
#   3) Run .describe() on yaw/pitch/roll for each.
#   4) Any missing values? df[["yaw","pitch","roll"]].isna().sum()
# Already notice a big difference between the two files? Jot it down in a comment.

# TODO: your code here


## Part 3 — Draw the target pattern of each file

Plot every `(targetX, targetY)` for **both** files so you can *see* the difference between the two formats. (Screen Y grows **downward**, so flip the y-axis to match the real screen.)

In [ ]:
# ✏️ Exercise 3 — plot the target pattern for BOTH files (figure it out)
# Make TWO scatter plots: targetX vs targetY for df_line, and for df_dots.
#   - look up plt.scatter, and how to flip the y-axis ("matplotlib invert y axis")
#   - put them side by side if you can (look up plt.subplots), and label axes + titles
# Then count distinct positions in each:
#   df[["targetX","targetY"]].drop_duplicates().shape[0]
# Questions (answer in a comment):
#   - Which file is the moving "with line" path, and which is the "without line" dots?
#   - How many distinct positions does each have, and why are they so different?

# TODO: your code here


## Part 4 — Does head pose follow the target? (compare both)

This is the heart of calibration: turn your head right → `yaw` changes → the dot is further right (`targetX`). Tilt up/down → `pitch` changes → `targetY`.

A **correlation** (`.corr()`, a number from -1 to 1) measures how strongly two columns move together. Compute it for **both** files and compare.

In [ ]:
# ✏️ Exercise 4 — head pose vs target, for BOTH files (work it out)
#   - For each file, scatter yaw vs targetX, and pitch vs targetY.
#   - For each file, compute corr(yaw, targetX) and corr(pitch, targetY).
#     (hint: df["yaw"].corr(df["targetX"]))
# Questions (answer in a comment):
#   - Are the relationships strong in BOTH files (|corr| close to 1)?
#   - Is one file noticeably cleaner/tighter than the other? Which?
#   - One correlation may come out NEGATIVE — is that a problem, or just a
#     direction/sign thing? (hint: screen Y grows downward)

# TODO: your code here


## Part 5 — Spot the difference 🔍

Now put the two formats head-to-head. Build a small **comparison table** with one column per file and rows for the things you measured.

In [ ]:
# ✏️ Exercise 5 — build a comparison table (figure it out)
# For BOTH files, compute these and put them side by side (a dict or small DataFrame
# works great). Try to fill every row:
#   - number of rows (frames)
#   - number of distinct target positions
#   - % of screen width covered:  (targetX.max() - targetX.min()) / SCREEN_W * 100
#   - % of screen height covered: (targetY.max() - targetY.min()) / SCREEN_H * 100
#   - corr(yaw, targetX)
#   - corr(pitch, targetY)
# (SCREEN_W / SCREEN_H come from meta_line / meta_dots: meta["calibrationWidth"], etc.)
# Which format wins on each row? Note it in a comment.

# TODO: your code here


## Part 6 — Think it through 🤔 (write your answers below)

Use what your plots and table showed. There are no single "right" answers — explain your reasoning.

1. **Which format gives more data points, and why?** Does more data automatically mean a *better* calibration?
2. **Which covers more of the screen?** Why does covering the edges matter for calibration?
3. **Which relationship (`yaw↔targetX`) looked cleaner?** Why might the moving dot be smoother — or noisier — than fixed dots?
4. **Following the dot:** is it easier to hold still on a *fixed dot*, or to smoothly *chase a moving dot*? How might that change the data?
5. **The moving file has time-ordered frames; the dots file barely does.** What extra analysis does the *with-line* format let you do that *without-line* can't?
6. **If you were running the study, which format would you choose, and why?**

*(Double-click this cell to edit, and type your answers under each question.)*

## 🏁 Bonus — what does a *failed* calibration look like?

A third sample is included, `sample_calibration_bad.csv`, where something went wrong. Load it with `load_calib()` and compare it to your good files:
- Plot `yaw` vs `targetX` and compute the correlation.
- How is it different from your good files? What do you think went wrong (tracking never locked on, the person didn't follow the dot, bad lighting…)?

*(No starter code — use what you built above.)*

In [ ]:
# TODO: your bonus code here


## ✅ Wrap-up — what to turn in

1. Your **target-pattern plots** for both files (Part 3), each with axis labels + title.
2. Your **comparison table** (Part 5).
3. Written answers to the **Part 6 questions**.
4. Your **bonus** finding: how the failed calibration differed.

Keep all the code you wrote to get each answer.